# Prepare the data (Zacharie Houle)

This notebook implements the *Prepare the Data* section of the INSY 674 rubric (Aurélien Géron, *Hands-On Machine Learning*, Appendix B, p. 757) and responds directly to the empirical findings from `01_explore_data.ipynb`:

0. **Checklist notes, target computation, and claim-grouped split:**
   * Work on a copy of `data/raw/train.csv` (36,176 claims, 15 attributes)
   * Compute the row-level audit targets defined in Notebook 01: `ReserveShortfall = UltimateIncurredClaimCost - InitialIncurredCalimsCost` (primary dollar target) and `LogDevelopmentRatio = log(UltimateIncurredClaimCost / InitialIncurredCalimsCost)` (scale-normalized log target)
   * Split into `train` (70%), `val` (15%), and `test` (15%) using `GroupShuffleSplit` on `ClaimNumber` so the 12,560 rows sharing repeated claim IDs never cross between train and evaluation splits
1. **Data cleaning:**
   * Treat the specific data-quality anomalies flagged in Notebook 01: 75 rows with `HoursWorkedPerWeek > 168`, 16 rows with `0` weekly hours, 2 negative reporting lags (-0.67 days), 225 reporting lags over 365 days, and 4 rows with `Age == 13`, plus train-only quantile clipping via `QuantileOutlierClipper`
   * Impute the 56 missing `WeeklyWages` (0.15%), 49 missing `HoursWorkedPerWeek` (0.14%), and 22 missing `MaritalStatus` (0.06%) values with missingness indicator flags (`add_indicator=True`)
2. **Feature selection:**
   * Exclude `ClaimNumber` and future outcome columns (`UltimateIncurredClaimCost`, `ReserveShortfall`, `LogDevelopmentRatio`) from the predictor matrix $X$, while keeping `InitialIncurredCalimsCost` in $X$ because the adjuster's initial reserve is known at First Notice of Loss (FNOL)
   * Prune collinear features with $|r| \ge 0.95$ on the training split via `CollinearityDropper`
3. **Feature engineering and external data enrichment:**
   * Join historical US Medical Care CPI (`data/external/fred_medical_cpi_annual.csv`, FRED series `CPIMEDSL`, 1988 to 2005) on `AccidentYear` to express initial reserves and wages in constant 2005 dollars
   * Decompose UTC timestamps (`DateTimeOfAccident`, `DateReported`) and compute `ReportingLagDays` (`total_seconds() / 86400`), `hourly_wage`, `hours_per_day`, and `total_dependents`
   * Vectorize the 20,596 distinct `ClaimDescription` injury phrases (average 7 words) using `TfidfVectorizer` + `TruncatedSVD` (`ClaimTextSVDTransformer`)
   * Apply $\log(1+x)$ / `Yeo-Johnson` transforms to heavy-tailed monetary predictors and `KBinsDiscretizer` to `Age`
4. **Feature scaling and export:**
   * Standardize predictors via `StandardScaler` inside a single leak-free scikit-learn `Pipeline` and export `train.parquet`, `val.parquet`, and `test.parquet` to `data/processed/`.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import set_config
from sklearn.model_selection import GroupShuffleSplit

REPO_ROOT = Path("..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.preprocessing import (
    ClaimTextSVDTransformer,
    CollinearityDropper,
    QuantileOutlierClipper,
    build_preprocessing_pipeline,
    compute_targets,
    decompose_datetime_features,
    drop_uninformative_columns,
)

set_config(display="diagram")
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
RANDOM_STATE = 42

RAW_DATA_DIR = REPO_ROOT / "data" / "raw"
EXTERNAL_DATA_DIR = REPO_ROOT / "data" / "external"
PROCESSED_DATA_DIR = REPO_ROOT / "data" / "processed"

## 0. Checklist notes: target definition, working on copies, and claim-grouped splitting

### Row-level target variables ($Y$) vs. FNOL predictors ($X$)
As established in `01_explore_data.ipynb` and the project `README.md`, our goal is to audit the human claims adjuster's initial reserve (`InitialIncurredCalimsCost`) at First Notice of Loss (FNOL). For every claim $i$, we compute two supervised targets from the final outcome (`UltimateIncurredClaimCost`):
1. `ReserveShortfall` $= \text{UltimateIncurredClaimCost}_i - \text{InitialIncurredCalimsCost}_i$: the exact dollar amount by which the adjuster under-reserved ($>0$) or over-reserved ($<0$). In the full dataset, its median is $82.70 and its mean is $3,209.23 (range: -$663,152.27 to +$3,984,135.94).
2. `LogDevelopmentRatio` $= \ln(\text{UltimateIncurredClaimCost}_i / \text{InitialIncurredCalimsCost}_i)$: the scale-normalized log multiplier (mean 0.24, std 0.81), defined on all 36,176 rows because both costs are strictly positive everywhere.

Why `InitialIncurredCalimsCost` stays in $X$ while `UltimateIncurredClaimCost` is removed from $X$:
* At FNOL (day one), the human adjuster has just entered `InitialIncurredCalimsCost`. Because that figure is already recorded when our audit model scores the file, using `InitialIncurredCalimsCost` as an input feature in $X$ is not data leakage. In fact, the model needs to see the initial reserve to know whether a given injury was under-priced at $1,000 or already adequately reserved at $15,000.
* By contrast, `UltimateIncurredClaimCost` is only known months or years later when the claim closes. Leaving `UltimateIncurredClaimCost` inside $X$ would constitute future data leakage, so we remove `UltimateIncurredClaimCost`, `ReserveShortfall`, and `LogDevelopmentRatio` from $X$ before fitting the pipeline.

### Why we use a claim-grouped split (`GroupShuffleSplit` on `ClaimNumber`)
Notebook 01 discovered that 12,560 of the 36,176 rows share a `ClaimNumber` that appears more than once (29,456 unique IDs total). To guarantee that linked records with the same `ClaimNumber` never sit in both `train` and `val`/`test`, we split by `ClaimNumber` groups (70% train, 15% validation, 15% test) before computing any statistics.

In [ ]:
# 1. Load a copy of raw data and compute the row-level target variables
raw_df = pd.read_csv(RAW_DATA_DIR / "train.csv")
data_copy = compute_targets(raw_df.copy())

# 2. Claim-grouped split: 70% train, 15% val, 15% test grouped by ClaimNumber
gss_test = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=RANDOM_STATE)
train_val_idx, test_idx = next(gss_test.split(data_copy, groups=data_copy["ClaimNumber"]))
train_val_df = data_copy.iloc[train_val_idx].reset_index(drop=True)
test_df = data_copy.iloc[test_idx].reset_index(drop=True)

gss_val = GroupShuffleSplit(n_splits=1, test_size=0.15 / 0.85, random_state=RANDOM_STATE)
train_idx, val_idx = next(gss_val.split(train_val_df, groups=train_val_df["ClaimNumber"]))
train_df = train_val_df.iloc[train_idx].reset_index(drop=True)
val_df = train_val_df.iloc[val_idx].reset_index(drop=True)

# Verify zero ClaimNumber overlap across splits
overlap_tv = set(train_df["ClaimNumber"]).intersection(set(val_df["ClaimNumber"]))
overlap_tt = set(train_df["ClaimNumber"]).intersection(set(test_df["ClaimNumber"]))
print(f"Train: {train_df.shape} | Val: {val_df.shape} | Test: {test_df.shape}")
print(f"ClaimNumber overlap (Train vs Val): {len(overlap_tv)} | (Train vs Test): {len(overlap_tt)}")

## 1. Data cleaning

### 1.1 Fix or remove outliers and recording anomalies: rationale and trade-offs

Notebook 01 identified four specific recording anomalies in the dataset without deleting them:
1. `HoursWorkedPerWeek`: 75 rows exceed 168 hours per week (up to 640 hours, likely monthly hours entered by mistake), and 16 rows record `0` hours despite having positive weekly wages.
2. `ReportingLagDays`: 2 rows have a slightly negative lag (-0.67 days due to timestamp rounding) and 225 rows exceed 365 days (up to 1,088.5 days).
3. `Age`: 4 rows record `Age == 13` (just below the 14 to 100 screening range).
4. Heavy right tails in `InitialIncurredCalimsCost` (median $2,000, max $830,000) and `WeeklyWages` (median $393.37, max $7,497).

Why we sanitize impossible values to `NaN` (plus indicator flags) instead of dropping rows:
* Dropping rows is rejected because an audit model in production must score every incoming FNOL claim, even if a clerk typed `400` in the weekly hours box.
* For the 75 rows with `HoursWorkedPerWeek > 168` and 16 rows with `HoursWorkedPerWeek <= 0`, dividing `WeeklyWages` by those raw numbers produces distorted hourly wages ($0.50/hr or division by zero). We therefore flag them with a binary column (`invalid_hours_flag = 1`) and set the implausible hour values to `NaN` so our train-fitted imputer reconstructs a realistic work week before computing `ApproxHourlyWage`.
* For the 2 negative reporting lags (-0.67 days), we flag them and clip lower bound at `0.0`, while adding a binary `reporting_lag_over_1yr` flag for the 225 delayed claims.
* For continuous heavy tails, `QuantileOutlierClipper` learns 1st and 99th percentile caps strictly on `train_df` and exposes `clip_outliers=True/False` as a hyperparameter for Notebook 04.

In [ ]:
# Empirical check of the anomalies on train_df before cleaning
num_preview_cols = [
    "Age", "DependentChildren", "DependentsOther", "WeeklyWages",
    "HoursWorkedPerWeek", "DaysWorkedPerWeek", "InitialIncurredCalimsCost"
]
outlier_audit = train_df[num_preview_cols].quantile([0.0, 0.01, 0.50, 0.99, 1.0]).T
outlier_audit.columns = ["min", "p01", "median", "p99", "max"]
outlier_audit

### 1.2 Fill in missing values: rationale and trade-offs

Notebook 01 showed that missingness is concentrated in three columns across the 36,176 claims: `WeeklyWages` (56 missing, 0.15%), `HoursWorkedPerWeek` (49 missing, 0.14%, plus the invalid hours converted to `NaN` above), and `MaritalStatus` (22 missing, 0.06%).

1. Why we add a boolean missingness indicator (`add_indicator=True`): As taught in Module 3 (§2.4), missingness in workers' compensation claims is rarely completely random (MCAR). Unreported wages or hours often signal casual, seasonal, or disputed employment, which can correlate with claim complexity.
2. Why we prefer `median` and `IterativeImputer` over `mean`: Because `WeeklyWages` is right-skewed (mean $416.47 vs. median $393.37), mean imputation pulls missing values upward. `SimpleImputer(strategy="median")` is robust to skew, while `IterativeImputer` (Module 2) predicts missing wages/hours from `PartTimeFullTime`, `DaysWorkedPerWeek`, and `Age`. Both are parameterized in `build_preprocessing_pipeline` so Notebook 04 can test them in cross-validation.
3. Categorical missing values (`MaritalStatus`): Imputed with the training mode (`most_frequent`) inside the categorical pipeline.

In [ ]:
# Missing value summary on train_df
missing_summary = pd.DataFrame({
    "missing_count": train_df.isna().sum(),
    "missing_pct": (train_df.isna().mean() * 100).round(3),
    "dtype": train_df.dtypes.astype(str),
})
missing_summary[missing_summary["missing_count"] > 0].sort_values("missing_pct", ascending=False)

## 2. Feature selection

### 2.1 Drop attributes that provide no useful information: rationale and trade-offs

Following Module 4 (*Dimensionality Reduction & Feature Selection*), we choose feature selection over full-table PCA for our structured attributes so actuaries and claims managers can still interpret individual predictors (such as reporting lag or initial reserve).

We apply two selection rules:
1. Exclude `ClaimNumber` (administrative identifier with no generalizable signal) and all outcome variables (`UltimateIncurredClaimCost`, `ReserveShortfall`, `LogDevelopmentRatio`) from $X$.
2. Apply `CollinearityDropper(threshold=0.95)` at the end of the pipeline, fitted strictly on `X_train`, to drop any redundant feature whose absolute correlation with another feature exceeds 0.95.

In [ ]:
OUTCOME_AND_ID_COLS = [
    "ClaimNumber",
    "UltimateIncurredClaimCost",
    "ReserveShortfall",
    "LogDevelopmentRatio",
]

X_train_raw = drop_uninformative_columns(train_df, OUTCOME_AND_ID_COLS)
X_val_raw = drop_uninformative_columns(val_df, OUTCOME_AND_ID_COLS)
X_test_raw = drop_uninformative_columns(test_df, OUTCOME_AND_ID_COLS)
print(f"Candidate predictor columns in X_train_raw ({X_train_raw.shape[1]}): {list(X_train_raw.columns)}")

## 3. Feature engineering and external data enrichment

Below we implement and justify each transformation recommended in Notebook 01 (§1.5, *Identify promising transformations* and *Identify extra data that would be useful*).

### 3.1 External data enrichment: FRED Medical Care CPI (`CPIMEDSL`, 1988 to 2005)
Claims in this dataset span 18 years (1988 to 2005). Because medical and wage inflation more than doubled over that window (US BLS Medical Care CPI rose from 138.6 in 1988 to 323.2 in 2005), a $2,000 initial reserve in 1988 represents a much more severe injury than a $2,000 reserve in 2005. Joining `data/external/fred_medical_cpi_annual.csv` on `AccidentYear` allows us to compute inflation-adjusted constant-2005-dollar features (`InitialCost_Real2005` and `WeeklyWages_Real2005`) without any look-ahead leakage, because the historical CPI of the accident year is known at FNOL.

### 3.2 Decompose features (UTC timestamps, categoricals, and `ClaimDescription` text)
1. Timestamps: We parse `DateTimeOfAccident` and `DateReported` in UTC and decompose them into `year`, `month`, `day`, `dayofweek`, `hour`, and `is_weekend`. As noted in Notebook 01, accident hour and day of week capture shift patterns, while accident year captures long-term policy drift.
2. Categoricals (`Gender`, `MaritalStatus`, `PartTimeFullTime`): Per the Module 3 encoding decision table (§2.3.12), these are low-cardinality nominal attributes with no natural ranking, so we use `OneHotEncoder`. Because Notebook 01 found a single row with `Gender == "U"`, we set `min_frequency=10` and `handle_unknown="infrequent_if_exist"` so rare or unseen labels are grouped safely without creating a near-zero-variance column.
3. Free text (`ClaimDescription`): Notebook 01 showed that `ClaimDescription` averages 7 words across 20,596 distinct strings (such as `"SLIPPED ON ROLLER TENDONITIS RIGHT SHOULDER"`). One-hot encoding 20,596 phrases would cause severe dimensionality explosion. Instead, our custom `ClaimTextSVDTransformer` fits `TfidfVectorizer(max_features=500, ngram_range=(1, 2))` followed by `TruncatedSVD(n_components=15)` on `train` only, extracting 15 dense injury-mechanism concepts.

### 3.3 Aggregate features into promising new features
1. `ReportingLagDays`: Exact elapsed days `(DateReported - DateTimeOfAccident).dt.total_seconds() / 86400` (with negative values clipped to 0 and flags for negative and >365-day lags). Delayed reporting is a strong indicator of soft-tissue or cumulative injuries whose costs escalate after FNOL.
2. `ApproxHourlyWage`: `WeeklyWages / HoursWorkedPerWeek` (computed after sanitizing hours outside `(0, 168]` to `NaN`).
3. `HoursPerDay`: `HoursWorkedPerWeek / DaysWorkedPerWeek` (shift length intensity).
4. `TotalDependents`: `DependentChildren + DependentsOther`.
5. `InitialReserveToWageRatio`: `InitialIncurredCalimsCost / WeeklyWages`, measuring how many weeks of pay the adjuster's initial reserve represents.

### 3.4 Mathematical transformations ($\log(1+x)$) and discretization (`KBinsDiscretizer`)
Monetary and lag features (`InitialIncurredCalimsCost`, `InitialCost_Real2005`, `WeeklyWages`, `WeeklyWages_Real2005`, `ApproxHourlyWage`, `InitialReserveToWageRatio`, `ReportingLagDays`) have strong positive skewness. Applying `log1p` compresses extreme right tails so linear and neural models are not dominated by outliers. Meanwhile, `Age` is discretized into 5 quantile brackets via `KBinsDiscretizer` (in addition to continuous `Age`) to capture non-linear recovery plateaus among older workers.

In [ ]:
cpi_df = pd.read_csv(EXTERNAL_DATA_DIR / "fred_medical_cpi_annual.csv")
display(cpi_df.head())

def engineer_features(df: pd.DataFrame, external_cpi: pd.DataFrame) -> pd.DataFrame:
    """Stateless feature cleaning, external CPI join, decomposition, and aggregation on a copy of df."""
    out = df.copy()

    # 1. Parse timestamps in UTC and compute exact ReportingLagDays as in Notebook 01
    acc_dt = pd.to_datetime(out["DateTimeOfAccident"], errors="coerce", utc=True)
    rep_dt = pd.to_datetime(out["DateReported"], errors="coerce", utc=True)
    raw_lag = (rep_dt - acc_dt).dt.total_seconds() / 86400.0

    out["negative_lag_flag"] = (raw_lag < 0).astype(int)
    out["lag_over_1yr_flag"] = (raw_lag > 365).astype(int)
    out["ReportingLagDays"] = raw_lag.clip(lower=0.0)
    out["AccidentYear"] = acc_dt.dt.year

    # 2. Join external FRED Medical Care CPI by AccidentYear
    out = out.merge(external_cpi, on="AccidentYear", how="left")
    out["CPI_Multiplier_to_2005"] = out["CPI_Multiplier_to_2005"].fillna(1.0)
    out["InitialCost_Real2005"] = out["InitialIncurredCalimsCost"] * out["CPI_Multiplier_to_2005"]
    out["WeeklyWages_Real2005"] = out["WeeklyWages"] * out["CPI_Multiplier_to_2005"]

    # 3. Decompose DateTimeOfAccident and DateReported into calendar/time components
    out = decompose_datetime_features(
        out, date_cols=["DateTimeOfAccident", "DateReported"], drop_original=True
    )

    # 4. Sanitize implausible HoursWorkedPerWeek (<= 0 or > 168) flagged in Notebook 01
    invalid_hours = (~out["HoursWorkedPerWeek"].between(0.001, 168.0)) & out["HoursWorkedPerWeek"].notna()
    out["invalid_hours_flag"] = invalid_hours.astype(int)
    out.loc[invalid_hours, "HoursWorkedPerWeek"] = np.nan

    # Flag Age outside 14-100 screening range (4 rows with Age == 13 in Notebook 01)
    out["age_under_14_flag"] = (out["Age"] < 14).astype(int)

    # 5. Domain ratios and aggregations
    out["ApproxHourlyWage"] = out["WeeklyWages"] / out["HoursWorkedPerWeek"]
    days_clean = out["DaysWorkedPerWeek"].replace(0, np.nan)
    out["HoursPerDay"] = out["HoursWorkedPerWeek"] / days_clean
    out["TotalDependents"] = out["DependentChildren"].fillna(0) + out["DependentsOther"].fillna(0)
    wages_pos = out["WeeklyWages"].where(out["WeeklyWages"] > 0, np.nan)
    out["InitialReserveToWageRatio"] = out["InitialIncurredCalimsCost"] / wages_pos

    return out

X_train_eng = engineer_features(X_train_raw, cpi_df)
X_val_eng = engineer_features(X_val_raw, cpi_df)
X_test_eng = engineer_features(X_test_raw, cpi_df)
print(f"Engineered predictor columns before ColumnTransformer: {X_train_eng.shape[1]}")

In [ ]:
# Empirical verification of skewness reduction on train_df before vs. after log1p
SKEWED_NUMERIC_COLS = [
    "InitialIncurredCalimsCost",
    "InitialCost_Real2005",
    "WeeklyWages",
    "WeeklyWages_Real2005",
    "ApproxHourlyWage",
    "InitialReserveToWageRatio",
    "ReportingLagDays",
]
skew_check = pd.DataFrame({
    "raw_skewness": X_train_eng[SKEWED_NUMERIC_COLS].skew().round(2),
    "log1p_skewness": np.log1p(X_train_eng[SKEWED_NUMERIC_COLS].clip(lower=0)).skew().round(2),
})
skew_check

## 4. Feature scaling and complete scikit-learn pipeline

Following Module 3 (§2.1 and §2.2), we compare the three scaling strategies below:

| Method | Formula | When to use | Why chosen or rejected here |
|---|---|---|---|
| `MinMaxScaler` | $(x - \min) / (\max - \min)$ | Bounded inputs with fixed range (such as image pixels 0 to 255) | Rejected as default because a single extreme wage or reserve crushes normal observations into a narrow band near 0. |
| `StandardScaler` | $(x - \mu) / \sigma$ | Unbounded continuous variables (especially after `log1p` transformation) | Chosen as default so regularized linear models (Ridge/Lasso), SVMs, and neural nets receive zero-mean, unit-variance inputs. |
| `RobustScaler` | $(x - \text{median}) / \text{IQR}$ | Heavy-tailed variables when outlier clipping is disabled | Exposed as a tunable hyperparameter (`scaler_type="robust"`) for Notebook 04. |

All steps (quantile clipping, missingness indicators, imputation, `log1p` transforms, `Age` discretization, one-hot encoding, TF-IDF + SVD text extraction, scaling, and collinearity filtering at $|r| \ge 0.95$) are combined inside `build_preprocessing_pipeline`.

In [ ]:
TEXT_COL = "ClaimDescription"
DISCRETIZE_COLS = ["Age"]
LOW_CARD_CAT_COLS = ["Gender", "MaritalStatus", "PartTimeFullTime"]
NUMERIC_COLS = [
    c for c in X_train_eng.select_dtypes(include=[np.number]).columns
    if c not in SKEWED_NUMERIC_COLS
]

prep_pipeline = build_preprocessing_pipeline(
    numeric_cols=NUMERIC_COLS,
    skewed_numeric_cols=SKEWED_NUMERIC_COLS,
    discretize_cols=DISCRETIZE_COLS,
    low_card_cat_cols=LOW_CARD_CAT_COLS,
    text_col=TEXT_COL,
    clip_outliers=True,
    num_imputer_strategy="median",
    add_missing_indicator=True,
    skew_transform="log1p",
    n_bins=5,
    n_text_svd_components=15,
    scaler_type="standard",
    collinearity_threshold=0.95,
)

prep_pipeline

## 5. Fit on train, transform all splits, and export to `data/processed/`

We fit `prep_pipeline` strictly on `X_train_eng` (`fit_transform`), transform `X_val_eng` and `X_test_eng` (`transform`), and attach the target and benchmark columns at the end of each table so Notebooks 03 and 04 can:
1. Train directly on `ReserveShortfall` (dollar target) or `LogDevelopmentRatio` (scale-normalized log target)
2. Convert log predictions back to dollars using `raw_InitialIncurredCalimsCost` and evaluate dollar MAE/RMSE against `raw_UltimateIncurredClaimCost`.

In [ ]:
X_train_prep = prep_pipeline.fit_transform(X_train_eng)
X_val_prep = prep_pipeline.transform(X_val_eng)
X_test_prep = prep_pipeline.transform(X_test_eng)

dropped_collinear = prep_pipeline.named_steps["collinearity_filter"].cols_to_drop_
print(f"Collinear columns dropped (|r| >= 0.95 on train): {dropped_collinear}")
print(f"Final prepared feature count (X): {X_train_prep.shape[1]}")

def attach_targets_and_export(X_prep: pd.DataFrame, split_df: pd.DataFrame, out_path: Path) -> pd.DataFrame:
    out = X_prep.copy()
    out["ReserveShortfall"] = split_df["ReserveShortfall"].values
    out["LogDevelopmentRatio"] = split_df["LogDevelopmentRatio"].values
    out["raw_InitialIncurredCalimsCost"] = split_df["InitialIncurredCalimsCost"].values
    out["raw_UltimateIncurredClaimCost"] = split_df["UltimateIncurredClaimCost"].values
    out.to_parquet(out_path, index=False)
    return out

train_out = attach_targets_and_export(X_train_prep, train_df, PROCESSED_DATA_DIR / "train.parquet")
val_out = attach_targets_and_export(X_val_prep, val_df, PROCESSED_DATA_DIR / "val.parquet")
test_out = attach_targets_and_export(X_test_prep, test_df, PROCESSED_DATA_DIR / "test.parquet")

print(f"Exported processed datasets to {PROCESSED_DATA_DIR}:")
print(f"  train.parquet: {train_out.shape}")
print(f"  val.parquet:   {val_out.shape}")
print(f"  test.parquet:  {test_out.shape}")